# Customer Intelligence & Support Copilot
## Instructor-led end-to-end notebook

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Before class
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

Teaching point: **fetching data** does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (
    (ROOT / "generate_data.py").exists()
    and (ROOT / "data" / "raw" / "customers.csv").exists()
) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: C:\Users\Star\Documents\Swiggy Instamart Data kit\Swiggy Instamart Data kit
Root files: ['.gitignore', 'GITHUB_RESOURCES.md', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [2]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customer_features.csv', 'customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [3]:
from pathlib import Path

raw_dir = ROOT / "data" / "raw"
if not (raw_dir / "customers.csv").exists():
    raise FileNotFoundError(
        f"Could not find data/raw/customers.csv. Notebook working directory: {Path.cwd()}"
    )

print("Reading data from:", raw_dir)

Reading data from: C:\Users\Star\Documents\Swiggy Instamart Data kit\Swiggy Instamart Data kit\data\raw


In [4]:
import pandas as pd

customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [
    ("customers", customers_raw),
    ("products", products_raw),
    ("orders", orders_raw),
    ("tickets", tickets_raw),
]:
    print(f"\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(
            lambda col: col.str.strip().eq("").sum()
        ),
        "missing_pct": (df.isna().mean() * 100).round(2),
    }))


customers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0



products: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0



tickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [5]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "data_pipeline.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "src" / "data_pipeline.py").exists():
    raise FileNotFoundError("Could not locate the project folder.")

sys.path.insert(0, str(ROOT))

from src.data_pipeline import audit, clean_data, validate
audit()


Auditing raw CSV files...
 - customers.csv: shape=(1200, 5)
 - orders.csv: shape=(13084, 9)
 - products.csv: shape=(40, 4)
 - tickets.csv: shape=(164, 5)


In [6]:
from src.data_pipeline import audit, clean_data, validate
audit()

Auditing raw CSV files...
 - customers.csv: shape=(1200, 5)
 - orders.csv: shape=(13084, 9)
 - products.csv: shape=(40, 4)
 - tickets.csv: shape=(164, 5)


## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [7]:
import importlib
from src import data_pipeline

data_pipeline = importlib.reload(data_pipeline)
clean = data_pipeline.clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
data_pipeline.validate()

Data cleaned and loaded into SQLite successfully.
customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35.0,Male
1,C0002,Delhi,2024-02-17,31.0,Male
2,C0003,Bengaluru,2025-02-18,18.0,NaN


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
5243,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
4370,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
11679,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN


tickets (164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


Validation passed: customers=1200, orders=12309


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [8]:
import sqlite3
db_path = ROOT / "data" / "instamart.db"
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    q2 = pd.read_sql_query("""
    SELECT c.city, p.category, SUM(o.order_amount) AS revenue
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    JOIN products AS p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY c.city, p.category
    ORDER BY c.city, p.category
""", con)

display(q2)
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

,name
0,customers
1,orders
2,products
3,tickets


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The lapse model uses orders before 2025-08-01 as features and August-September orders only to define the label. It adds RFM interactions, age and signup tenure, imputes missing values, one-hot encodes city and gender, and scales numeric inputs for Logistic Regression. Random Forest feature selection and GridSearchCV tuning run inside the training pipeline. Logistic Regression and Random Forest baselines are compared with a majority-class baseline; the final F1 threshold is selected from out-of-fold training predictions and evaluated on the held-out test set. Accuracy is data-dependent, and R² is not appropriate for this classification task.

In [9]:
import importlib
import src.customer_models as customer_models
customer_models = importlib.reload(customer_models)
rfm_and_segments = customer_models.rfm_and_segments
build_lapse_model = customer_models.build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments()
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

,cluster,count,mean_recency,mean_frequency,mean_monetary
0,0,417,8.329,16.192,2876.436
1,1,177,98.339,1.847,588.767
2,2,437,24.941,3.339,956.681
3,3,125,22.768,5.944,6613.503


,cluster,count,mean_recency,mean_frequency,mean_monetary
0,0,417,8.33,16.19,2876.44
1,1,177,98.34,1.85,588.77
2,2,437,24.94,3.34,956.68
3,3,125,22.77,5.94,6613.50


PCA explained variance ratio: [0.62094188 0.2301486 ] total: 0.8510904775549722


,customer_id,recency,frequency,monetary,cluster,pca_1,pca_2
0,C0001,33,2,378.00,2,-1.051909,-0.471754
1,C0002,22,6,3946.90,3,0.373324,0.621337
2,C0003,1,20,3264.05,0,1.857033,-0.402242
3,C0004,42,3,272.25,2,-1.139868,-0.409763
4,C0005,117,2,324.00,1,-2.505681,0.764970


{'best_params': {'max_depth': 5, 'n_estimators': 100},
 'accuracy': 0.7920353982300885,
 'roc_auc': 0.7853898096992019,
 'train_samples': 901,
 'test_samples': 226,
 'churn_rate': 0.19875776397515527}

### Instructor discussion: leakage
Ask students to identify:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [10]:
import importlib
from src import triage

triage = importlib.reload(triage)
triage_results, triage_summary = triage.run_pipeline(prompt=triage.FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results["risk"].value_counts().rename_axis("risk").reset_index(name="tickets"))

Triage completed successfully across 164 tickets.


,ticket_id,category,risk
0,T0001,billing,low
1,T0002,fraud,high
2,T0003,delivery,medium


,risk,tickets
0,low,1
1,high,1
2,medium,1


### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [11]:
import importlib
from src import policy_rag

policy_rag = importlib.reload(policy_rag)
rag = policy_rag.PolicyRAG()
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
question = "What delivery time makes a completed order eligible for a full refund?"
display(pd.DataFrame(rag.retrieve(question)))
policy_eval = policy_rag.evaluate(rag)
display(policy_eval)
print("Expected-document matches:", int(policy_eval["match"].sum()))

Policy chunks: 5


,file,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
2,payment_policy.txt,"Payment Policy. Customers can pay using UPI, d..."
3,privacy_policy.txt,Privacy Policy. Customer phone numbers and ema...
4,refund_policy.txt,Refund Policy. A refund can be requested only ...


,file,text
0,refund_policy.txt,Refund Policy. A refund can be requested only ...


,question,retrieved_doc,match
0,What delivery time makes a completed order eli...,refund_policy.txt,True
1,Who must approve a refund above INR 1000?,refund_policy.txt,True
2,How many days do I have to raise a refund requ...,refund_policy.txt,True
3,What is the delivery fee on small orders?,delivery_policy.txt,True
4,On orders of what value is delivery free?,delivery_policy.txt,True
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True
6,Is cash on delivery available for a large order?,payment_policy.txt,True
7,How soon is money returned when a payment fail...,payment_policy.txt,True
8,Can wallet balance be refunded as cash?,payment_policy.txt,True
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True


Expected-document matches: 12


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [12]:
import importlib
from src import refund_agent

refund_agent = importlib.reload(refund_agent)
print("Available refund tool: get_order")
with sqlite3.connect(refund_agent.DB_PATH) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print("Example order lookup:", example_order, refund_agent.get_order(example_order))

Available refund tool: get_order
Example order lookup: ORD-111469 {"order_id": "ORD-111469", "customer_id": "C1122", "product_id": "P017", "quantity": 3, "order_ts": "2025-04-01 08:26:52", "status": "completed", "discount_pct": 25, "order_amount": 45.0, "delivery_minutes": 72.0}


## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [13]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['churn_model.pkl', 'customer_rfm_segments.csv', 'customer_segments.csv', 'customers_clean.csv', 'lapse_model.joblib', 'orders_clean.csv', 'products_clean.csv', 'retention_list.csv', 'scaler.pkl', 'tickets_clean.csv', 'triage_results.csv']
\nstate
[]
Database exists: True


In [14]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

for filename in [
    "customer_segments.csv",
    "retention_list.csv",
    "triage_results.csv",
]:
    print(f"\n{filename}")
    display(pd.read_csv(ROOT / "outputs" / filename))


customer_segments.csv


,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,At Risk,False
1,C0002,28,7,4644.90,1,Premium,True
2,C0003,7,22,3644.85,2,Frequent,False
3,C0004,0,5,369.00,0,At Risk,False
4,C0005,147,2,324.00,3,At Risk,False
5,C0006,34,1,90.00,0,At Risk,False
6,C0007,3,18,2600.00,2,Frequent,False
7,C0008,3,15,3096.50,2,Frequent,False
8,C0009,19,11,8424.80,1,Premium,True
9,C0010,66,4,450.00,0,At Risk,False



retention_list.csv


,customer_id,lapse_probability,actually_lapsed
0,C0862,0.510513,0
1,C0267,0.477212,0
2,C1008,0.473260,0
3,C0060,0.471746,0
4,C0802,0.471582,0
5,C0942,0.465245,0
6,C0553,0.462799,0
7,C0021,0.461252,0
8,C0133,0.460743,0
9,C0565,0.458271,1



triage_results.csv


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,At Risk,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,NaN,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At Risk,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,78
